## Setup & Imports

In [ ]:
# TODO: Your code here

# Step 1: Convert the list of probability arrays from multilabel_model.estimators_ into a single probability matrix via np.column_stack.
# Step 2: Feed that matrix into label_ranking_average_precision_score together with Y_test_mov.
# Step 3: Explain what the LRAP value means for this dataset.


## Utility Functions (Pre-filled - Run This Cell)

In [ ]:
def plot_confusion_matrix_heatmap(cm, labels=None, title='Confusion Matrix'):
    """Visualize confusion matrix"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

def plot_class_distribution(y, title='Class Distribution'):
    """Visualize class distribution"""
    plt.figure(figsize=(8, 5))
    if isinstance(y, pd.Series):
        y_counts = y.value_counts().sort_index()
    else:
        y_counts = pd.Series(y).value_counts().sort_index()
    
    plt.bar(y_counts.index, y_counts.values, alpha=0.7, edgecolor='black')
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title(title)
    
    total = y_counts.sum()
    for idx in y_counts.index:
        v = y_counts[idx]
        plt.text(idx, v + total*0.01, f'{v}\n({100*v/total:.1f}%)', 
                ha='center', va='bottom')
    plt.tight_layout()
    plt.show()

def compare_models_metrics(metrics_dict):
    """Create comparison table"""
    return pd.DataFrame(metrics_dict).T.round(3)

print("✓ Utility functions loaded")

---

# Task 1: Imbalanced Binary Classification (⏱️ ~40 min)

**Scenario:** Build a spam detector for an email service. Dataset has severe class imbalance (95% ham, 5% spam).

**Business Context:**
- False Positive (ham → spam): Very bad - user misses important email ($10 cost)
- False Negative (spam → ham): Less bad - user deletes spam manually ($1 cost)

**You'll Learn:**
- Why accuracy fails for imbalanced data
- Random Oversampling vs Class Weights vs SMOTE
- ROC-AUC and Precision-Recall curves
- Threshold optimization for business objectives
- Proper cross-validation with resampling

## 1.1) Load & Explore Dataset (Pre-filled)

In [ ]:
from sklearn.datasets import make_classification

# Generate imbalanced spam detection dataset
X_spam, y_spam = make_classification(
    n_samples=5000,
    n_features=30,
    n_informative=20,
    n_redundant=5,
    n_classes=2,
    weights=[0.95, 0.05],  # 95% ham, 5% spam
    flip_y=0.02,
    random_state=RANDOM_STATE
)

print(f"Dataset shape: {X_spam.shape}")
print(f"\nClass distribution:")
print(pd.Series(y_spam).value_counts())
print(f"\nImbalance ratio: {pd.Series(y_spam).value_counts()[0] / pd.Series(y_spam).value_counts()[1]:.1f}:1")

plot_class_distribution(y_spam, title='Spam vs Ham Distribution')

## 1.2) Prepare Data (Pre-filled)

In [ ]:
# Train-test split
X_train_spam, X_test_spam, y_train_spam, y_test_spam = train_test_split(
    X_spam, y_spam, test_size=0.25, stratify=y_spam, random_state=RANDOM_STATE
)

# Scale features
scaler_spam = StandardScaler()
X_train_spam_scaled = scaler_spam.fit_transform(X_train_spam)
X_test_spam_scaled = scaler_spam.transform(X_test_spam)

print(f"Training set: {X_train_spam_scaled.shape}")
print(f"Test set: {X_test_spam_scaled.shape}")
print(f"\nTrain class distribution:")
print(pd.Series(y_train_spam).value_counts())

## 1.3) Baseline Model ✏️ TODO (⏱️ ~5 min)

**Your Task:**
1. Train LogisticRegression on imbalanced data
2. Predict on test set
3. Calculate: accuracy, precision, recall, F1, ROC-AUC
4. Print classification report and confusion matrix

**Key Question:** Why is 95% accuracy misleading here?

In [ ]:
# TODO: Your code here

# Step 1: Instantiate LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) and fit it on X_train_spam_scaled / y_train_spam.
# Step 2: Use predict and predict_proba on X_test_spam_scaled to capture both labels and positive-class probabilities.
# Step 3: Compute accuracy, precision, recall, F1, and ROC-AUC (store them in a dictionary for easy comparison).
# Step 4: Print classification_report with target names ['Ham', 'Spam'] and display the confusion matrix via plot_confusion_matrix_heatmap.


## 1.4) Random Oversampling ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Use `RandomOverSampler` to balance training data
2. Train model on resampled data
3. Evaluate on original test set (never resample test!)
4. Compare metrics with baseline

**Key Point:** Oversampling duplicates minority samples. Test set remains unchanged!

In [ ]:
# TODO: Your code here

# Step 1: Use RandomOverSampler(random_state=RANDOM_STATE) to resample X_train_spam_scaled / y_train_spam and print the resulting class counts.
# Step 2: Train LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) on the resampled data.
# Step 3: Evaluate on the untouched X_test_spam_scaled to compute accuracy, precision, recall, F1, and ROC-AUC.
# Step 4: Show the classification_report and confusion matrix using plot_confusion_matrix_heatmap.


## 1.5) SMOTE (Synthetic Oversampling) ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Apply SMOTE to generate synthetic minority samples
2. Train model on SMOTE data
3. Evaluate and compare with ROS

**Key Difference:** SMOTE creates *synthetic* samples by interpolating between neighbors, not just duplicating.

In [ ]:
# TODO: Your code here

# Step 1: Create SMOTE(random_state=RANDOM_STATE) and resample X_train_spam_scaled / y_train_spam, then print the balanced class counts.
# Step 2: Train LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) on the SMOTE-balanced data.
# Step 3: Evaluate on X_test_spam_scaled by capturing predictions plus positive-class probabilities.
# Step 4: Compute accuracy, precision, recall, F1, ROC-AUC, and show both the classification report and confusion matrix.


## 1.6) Class Weights

In [ ]:
# TODO: Your code here

# Step 1: Initialize LogisticRegression with class_weight='balanced' (keep max_iter=1000, random_state=RANDOM_STATE) and fit it on X_train_spam_scaled.
# Step 2: Evaluate on X_test_spam_scaled to obtain predictions and probabilities.
# Step 3: Calculate accuracy, precision, recall, F1, and ROC-AUC for this strategy.
# Step 4: Display the classification_report and confusion matrix so you can compare against ROS/SMOTE.


## 1.7) Compare All Approaches ✏️ TODO (⏱️ ~5 min)

**Your Task:**
1. Create comparison table with all 4 approaches
2. Plot ROC curves for all models
3. Plot Precision-Recall curves

**Question:** For spam detection, which metric matters most? Which model wins?

In [ ]:
# TODO: Your code here

# Step 1: Combine baseline_metrics, ros_metrics, smote_metrics, and weighted_metrics into a comparison DataFrame via compare_models_metrics.
# Step 2: Plot ROC curves for each model using the stored probability vectors (baseline, Random OS, SMOTE, class weights).
# Step 3: Plot precision-recall curves for the same probabilities and annotate each line with its AUC / average precision.


## 1.8) Threshold Optimization ✏️ TODO (⏱️ ~8 min)

**Business Goal:** Minimize cost = FP × $10 + FN × $1

**Your Task:**
1. Use best model (probably weighted or SMOTE)
2. Try thresholds from 0.1 to 0.9
3. Calculate business cost at each threshold
4. Find optimal threshold
5. Evaluate model at optimal threshold

In [ ]:
# TODO: Your code here

# Step 1: Write a helper calculate_cost that converts confusion_matrix counts into FP/FN costs (FP=$10, FN=$1).
# Step 2: Loop over thresholds from 0.10 to 0.90 (step 0.05) using y_pred_proba_weighted to create binary predictions.
# Step 3: For each threshold, store cost, precision, recall, and F1 in a DataFrame so you can locate the minimum-cost threshold.
# Step 4: Plot metrics vs threshold on one subplot and business cost vs threshold on another, marking the optimal point.
# Step 5: Recompute classification_report at the best threshold to show the improved trade-offs.


## 1.9) Cross-Validation with SMOTE ✏️ TODO (⏱️ ~5 min)

**Critical:** Apply SMOTE *inside* each CV fold to avoid data leakage!

**Your Task:**
1. Implement 5-fold stratified CV
2. Apply SMOTE inside each fold
3. Calculate mean F1-score across folds

In [ ]:
# TODO: Your code here

# Step 1: Create a StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE) plus a SMOTE instance.
# Step 2: Within each fold, apply SMOTE to the training split only, train LogisticRegression(max_iter=1000), and score on the validation split.
# Step 3: Collect F1-scores for every fold and print the list along with its mean and standard deviation.


---

# Task 2: Multiclass Classification (⏱️ ~40 min)

**Scenario:** Classify handwritten digits (0-9) with artificial class imbalance.

**You'll Learn:**
- One-vs-Rest (OvR) strategy
- One-vs-One (OvO) strategy
- Macro vs Weighted F1-scores
- SMOTE for multiclass
- Per-class performance analysis

## 2.1) Load Digits Dataset (Pre-filled)

In [ ]:
from sklearn.datasets import load_digits

# Load digits dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# Create artificial imbalance: keep all of classes 0-5, but only 20% of classes 6-9
np.random.seed(RANDOM_STATE)
mask = (y_digits <= 5) | (np.random.rand(len(y_digits)) < 0.2)
X_digits_imb = X_digits[mask]
y_digits_imb = y_digits[mask]

print(f"Dataset shape: {X_digits_imb.shape}")
print(f"\nClass distribution:")
print(pd.Series(y_digits_imb).value_counts().sort_index())

plot_class_distribution(y_digits_imb, title='Digit Distribution (Imbalanced)')

## 2.2) Prepare Data (Pre-filled)

In [ ]:
# Train-test split
X_train_dig, X_test_dig, y_train_dig, y_test_dig = train_test_split(
    X_digits_imb, y_digits_imb, test_size=0.25, stratify=y_digits_imb, random_state=RANDOM_STATE
)

# Scale features
scaler_dig = StandardScaler()
X_train_dig_scaled = scaler_dig.fit_transform(X_train_dig)
X_test_dig_scaled = scaler_dig.transform(X_test_dig)

print(f"Training set: {X_train_dig_scaled.shape}")
print(f"Test set: {X_test_dig_scaled.shape}")

## 2.3) Baseline Multiclass Model ✏️ TODO (⏱️ ~5 min)

**Your Task:**
1. Train LogisticRegression (it uses OvR by default)
2. Evaluate with accuracy, macro F1, and weighted F1
3. Print classification report

**Key Metrics:**
- **Macro F1**: Average F1 across classes (treats all classes equally)
- **Weighted F1**: Weighted by class support (accounts for imbalance)

In [ ]:
# TODO: Your code here

# Step 1: Fit LogisticRegression(max_iter=2000, random_state=RANDOM_STATE) on X_train_dig_scaled / y_train_dig.
# Step 2: Predict labels for X_test_dig_scaled to obtain y_pred_dig_base.
# Step 3: Calculate accuracy, macro F1, and weighted F1, then print them for discussion.
# Step 4: Display classification_report and a confusion matrix heatmap for the baseline model.


## 2.4) Apply SMOTE for Multiclass ✏️ TODO (⏱️ ~7 min)

**Your Task:**
1. Apply SMOTE to balance all classes
2. Train model on balanced data
3. Compare macro and weighted F1 with baseline

**Note:** SMOTE works for multiclass - it balances all minority classes!

In [ ]:
# TODO: Your code here

# Step 1: Run SMOTE(random_state=RANDOM_STATE) on X_train_dig_scaled / y_train_dig and show the balanced class counts.
# Step 2: Train LogisticRegression(max_iter=2000, random_state=RANDOM_STATE) on the SMOTE output.
# Step 3: Evaluate on X_test_dig_scaled, collecting accuracy, macro F1, and weighted F1.
# Step 4: Print the classification_report and visualize the confusion matrix to compare with the baseline.


## 2.5) One-vs-Rest (OvR) Strategy ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Explicitly use `OneVsRestClassifier`
2. Train on SMOTE data
3. Evaluate and compare

**Concept:** OvR trains N binary classifiers (one per class). Each asks: "Is this sample class X or not?"

In [ ]:
# TODO: Your code here

# Step 1: Build OneVsRestClassifier(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)).
# Step 2: Train on the SMOTE-balanced training data (X_train_dig_smote / y_train_dig_smote).
# Step 3: Predict on X_test_dig_scaled and compute accuracy, macro F1, and weighted F1.
# Step 4: Print the metrics and classification_report, noting any per-class insights.


## 2.6) One-vs-One (OvO) Strategy ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Use `OneVsOneClassifier`
2. Train on SMOTE data
3. Evaluate and compare with OvR

**Concept:** OvO trains N×(N-1)/2 binary classifiers (one for each pair). For 10 classes, that's 45 classifiers!

In [ ]:
# TODO: Your code here

# Step 1: Instantiate OneVsOneClassifier(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)).
# Step 2: Train on X_train_dig_smote / y_train_dig_smote.
# Step 3: Evaluate on X_test_dig_scaled, capturing accuracy plus macro/weighted F1.
# Step 4: Print the metrics, classification_report, and optionally the number of binary estimators.


## 2.7) Compare Strategies ✏️ TODO (⏱️ ~4 min)

**Your Task:**
1. Create comparison table for all approaches
2. Analyze which strategy works best
3. Consider training time vs performance trade-offs

In [ ]:
# TODO: Your code here

# Step 1: Consolidate the metrics dictionaries for Baseline, SMOTE, OvR, and OvO into a DataFrame via compare_models_metrics.
# Step 2: Plot the comparison (e.g., bar chart) to visualize how each strategy performs across accuracy, macro F1, and weighted F1.
# Step 3: Summarize which approach balances performance versus complexity/training time.


## 2.8) Per-Class Analysis ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Extract F1-scores for each digit
2. Visualize which digits are hardest to classify
3. Identify confusion patterns

In [ ]:
# TODO: Your code here

# Step 1: Compute per-class F1-scores using the predictions from your best-performing multiclass model (e.g., OvR).
# Step 2: Plot a bar chart of F1 by digit to highlight strong vs. weak classes.
# Step 3: Annotate or discuss which digits need more attention and why.


## 2.9) Cross-Validation ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Perform 5-fold CV with SMOTE inside folds
2. Calculate mean macro F1-score

In [ ]:
# TODO: Your code here

# Step 1: Configure StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE) plus SMOTE(random_state=RANDOM_STATE).
# Step 2: Apply SMOTE within each fold's training data, train LogisticRegression(max_iter=2000), and score on the validation split.
# Step 3: Collect macro F1 for each fold and report the list with mean ± std.


---

# Task 3: Multilabel Classification (⏱️ ~35 min)

**Scenario:** Predict movie genres - a movie can have multiple genres simultaneously!

**You'll Learn:**
- Difference between multiclass and multilabel
- Hamming Loss and Subset Accuracy
- Label Ranking Average Precision (LRAP)
- Macro vs Micro vs Samples averaging

## 3.1) Generate Multilabel Dataset (Pre-filled)

In [ ]:
from sklearn.datasets import make_multilabel_classification

# Generate multilabel movie genre dataset
X_movies, Y_movies = make_multilabel_classification(
    n_samples=1000,
    n_features=50,
    n_classes=5,  # 5 genres
    n_labels=2,   # Average 2 genres per movie
    random_state=RANDOM_STATE
)

genres = ['Action', 'Comedy', 'Drama', 'Thriller', 'Romance']

print(f"Dataset shape: {X_movies.shape}")
print(f"Labels shape: {Y_movies.shape}")
print(f"\nFirst 5 movies:")
print(pd.DataFrame(Y_movies[:5], columns=genres))

# Label statistics
print(f"\nAverage labels per movie: {Y_movies.sum(axis=1).mean():.2f}")
print(f"\nLabel frequencies:")
print(pd.Series(Y_movies.sum(axis=0), index=genres))

## 3.2) Prepare Data (Pre-filled)

In [ ]:
# Train-test split
X_train_mov, X_test_mov, Y_train_mov, Y_test_mov = train_test_split(
    X_movies, Y_movies, test_size=0.25, random_state=RANDOM_STATE
)

# Scale features
scaler_mov = StandardScaler()
X_train_mov_scaled = scaler_mov.fit_transform(X_train_mov)
X_test_mov_scaled = scaler_mov.transform(X_test_mov)

print(f"Training set: {X_train_mov_scaled.shape}")
print(f"Test set: {X_test_mov_scaled.shape}")

## 3.3) Build Multilabel Classifier ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Use `OneVsRestClassifier` with LogisticRegression
2. Fit on training data
3. Make predictions

**Key Point:** In multilabel, each sample can have multiple labels! Y is a matrix, not a vector.

In [ ]:
# TODO: Your code here

# Step 1: Wrap LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) inside OneVsRestClassifier for multilabel training.
# Step 2: Fit the model on X_train_mov_scaled and Y_train_mov.
# Step 3: Predict labels (and keep probabilities) on X_test_mov_scaled for later metrics/visualizations.
# Step 4: Print a few sample true/predicted label rows to sanity-check results.


## 3.4) Multilabel Metrics ✏️ TODO (⏱️ ~8 min)

**Your Task:** Calculate multilabel-specific metrics:

1. **Hamming Loss**: Fraction of wrong labels (lower is better)
2. **Subset Accuracy**: Fraction with ALL labels correct (strict!)
3. **Macro F1**: Average F1 per label (treats labels equally)
4. **Micro F1**: Aggregate across all label-sample pairs
5. **Samples F1**: Average F1 per sample

**Question:** Why is subset accuracy usually very low?

In [ ]:
# TODO: Your code here

# Step 1: Compute Hamming Loss, Subset Accuracy, Macro F1, Micro F1, and Samples F1 using Y_test_mov vs Y_pred_mov.
# Step 2: Print the metrics, show classification_report per genre, and optionally visualize per-label F1 with a bar chart.


## 3.5) Label Ranking Average Precision ✏️ TODO (⏱️ ~5 min)

**Your Task:**
1. Calculate LRAP using predicted probabilities
2. Understand what this metric measures

**LRAP:** Measures how well true labels are ranked higher than false labels. Range: 0-1 (higher is better).

In [ ]:
# TODO: Your code here

# Step 1: Convert the list of probability arrays from multilabel_model.estimators_ into a single probability matrix via np.column_stack.
# Step 2: Feed that matrix into label_ranking_average_precision_score together with Y_test_mov.
# Step 3: Explain what the LRAP value means for this dataset.


## 3.6) Cross-Validation ✏️ TODO (⏱️ ~5 min)

**Your Task:**
1. Perform 5-fold CV for multilabel classification
2. Calculate mean Hamming Loss and Macro F1

In [ ]:
# TODO: Your code here

# Step 1: Build scorers for Hamming Loss (greater_is_better=False) and Macro F1 using make_scorer.
# Step 2: Run cross_val_score with 5-fold CV on the multilabel_model / movie data for each scorer (optionally using n_jobs=-1).
# Step 3: Print the mean ± std for both metrics.


## 3.7) Analyze Label Co-occurrence ✏️ TODO (⏱️ ~6 min)

**Your Task:**
1. Calculate how often genre pairs appear together
2. Visualize co-occurrence matrix

This helps understand label dependencies!

In [ ]:
# TODO: Your code here

# Step 1: Compute the label co-occurrence matrix via Y_movies.T @ Y_movies (zero out the diagonal).
# Step 2: Visualize the matrix with sns.heatmap, labeling rows/columns with genre names.
# Step 3: Highlight interesting genre pairs that frequently appear together.


---

# Summary & Reflection

**Congratulations!** You've completed all three advanced classification tasks. 🎉

## Key Takeaways

### Task 1: Imbalanced Binary Classification
- ✅ Accuracy is misleading for imbalanced data
- ✅ Random Oversampling duplicates minority samples
- ✅ SMOTE creates synthetic samples via interpolation
- ✅ Class weights adjust loss function (no resampling needed)
- ✅ Threshold optimization aligns with business objectives
- ✅ Proper CV requires resampling inside folds

### Task 2: Multiclass Classification  
- ✅ One-vs-Rest trains N classifiers (one per class)
- ✅ One-vs-One trains N×(N-1)/2 classifiers (all pairs)
- ✅ Macro F1 treats all classes equally
- ✅ Weighted F1 accounts for class imbalance
- ✅ SMOTE works for multiclass scenarios
- ✅ Per-class analysis reveals performance patterns

### Task 3: Multilabel Classification
- ✅ Multiple labels per instance (not mutually exclusive)
- ✅ Hamming Loss measures label-wise accuracy
- ✅ Subset Accuracy requires perfect label set match
- ✅ Macro/Micro/Samples averaging serve different purposes
- ✅ LRAP evaluates label ranking quality
- ✅ Label co-occurrence reveals dependencies

## Discussion Questions

1. **When to use ROS vs SMOTE vs Class Weights?**
   - Consider: dataset size, overfitting risk, computational cost

2. **OvR vs OvO trade-offs?**
   - Consider: number of classes, training time, interpretability

3. **What makes multilabel fundamentally different?**
   - Think: label independence, evaluation complexity, real-world applications

4. **Which averaging method (macro/micro/weighted) when?**
   - Consider: class imbalance, business priorities, reporting needs

## Next Steps

- Try different classifiers (Random Forest, XGBoost, Neural Networks)
- Explore other oversampling techniques (ADASYN, Borderline-SMOTE)
- Study under-sampling methods (Tomek Links, ENN)
- Learn about classifier chains for multilabel
- Experiment with ensemble methods

---

**Well done!** You now have practical experience with advanced classification scenarios.

---

# Bonus Challenges (⏱️ ~5 min)

## Bonus 1: Cost-Sensitive Threshold Analysis
Compare threshold optimization with custom class weights. Which gives better business outcomes?

## Bonus 2: Feature Importance
Use `model.coef_` to identify most important features for each class in multiclass classification.

## Bonus 3: Ensemble Voting
Combine predictions from ROS, SMOTE, and Weighted models using majority voting.

## Bonus 4: Classifier Chains
For multilabel, try `ClassifierChain` to model label dependencies.

## Bonus 5: Real Dataset
Apply techniques to a real imbalanced dataset (e.g., Credit Card Fraud from Kaggle).